# AnimateDiff — Make Any Image Fly

Generate a flying animation from a single image using **AnimateDiff** + **IP-Adapter** on Stable Diffusion 1.5.

- **Model**: SD 1.5 + AnimateDiff Motion Adapter + IP-Adapter
- **VRAM**: ~6GB (float16) — fits T4 free tier
- **Output**: 16 frames at 512×512, exported as MP4/GIF

Go to **Runtime > Change runtime type > T4 GPU**

In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors pillow imageio[ffmpeg] peft ip_adapter

import torch
print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory
    print(f'GPU: {props.name} — {vram / 1024**3:.1f} GiB')
else:
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4')
print('Dependencies installed.')

In [ ]:
# Cell 2: Load AnimateDiff pipeline
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from diffusers.utils import export_to_gif, export_to_video
import torch

# Load motion adapter
adapter = MotionAdapter.from_pretrained(
    'guoyww/animatediff-motion-adapter-v1-5-3',
    torch_dtype=torch.float16,
)

# Load SD 1.5 + motion adapter
pipe = AnimateDiffPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    motion_adapter=adapter,
    torch_dtype=torch.float16,
)

pipe.scheduler = DDIMScheduler.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    subfolder='scheduler',
    clip_sample=False,
    timestep_spacing='linspace',
    beta_schedule='linear',
    steps_offset=1,
)

pipe.enable_model_cpu_offload()
print('AnimateDiff pipeline loaded!')

In [ ]:
# Cell 3: Load character reference image
import urllib.request
from PIL import Image

IMG_URL = 'https://raw.githubusercontent.com/mangeshgwagle/attestor/main/training/character_ref.webp'
IMG_PATH = '/content/character_ref.webp'

print('Downloading character image...')
urllib.request.urlretrieve(IMG_URL, IMG_PATH)
ref_image = Image.open(IMG_PATH).convert('RGB').resize((512, 512))
print(f'Loaded: {ref_image.size}')
display(ref_image)

In [ ]:
# Cell 4: Load IP-Adapter for image-guided generation
from diffusers.utils import load_image

pipe.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder='models',
    weight_name='ip-adapter_sd15.bin',
)
pipe.set_ip_adapter_scale(0.6)
print('IP-Adapter loaded! The character\'s appearance will guide the animation.')

In [ ]:
# Cell 5: Generate flying animation
import torch

PROMPT = (
    'angelic warrior character flying through clouds, '
    'golden wings spread wide, glowing armor, epic celestial sky, '
    'dynamic flying pose, wind flowing through cape, '
    'fantasy game art style, high quality, detailed'
)
NEGATIVE = (
    'ugly, blurry, low quality, distorted, deformed, '
    'static, still, no motion, watermark, text'
)

generator = torch.manual_seed(42)

output = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE,
    ip_adapter_image=ref_image,
    num_frames=16,
    guidance_scale=7.5,
    num_inference_steps=25,
    generator=generator,
    width=512,
    height=512,
)

frames = output.frames[0]
print(f'Generated {len(frames)} frames!')

In [ ]:
# Cell 6: Export to MP4 and GIF, display inline
from diffusers.utils import export_to_video, export_to_gif
from IPython.display import HTML, Image as IPImage
from base64 import b64encode
import os

MP4_PATH = '/content/flying_character.mp4'
GIF_PATH = '/content/flying_character.gif'

export_to_video(frames, MP4_PATH, fps=8)
export_to_gif(frames, GIF_PATH)

mp4_size = os.path.getsize(MP4_PATH) / 1024**2
gif_size = os.path.getsize(GIF_PATH) / 1024**2
print(f'MP4: {mp4_size:.1f} MB | GIF: {gif_size:.1f} MB')

# Display MP4 inline
with open(MP4_PATH, 'rb') as f:
    mp4 = b64encode(f.read()).decode()
display(HTML(f'''
<video width="512" controls autoplay loop>
  <source src="data:video/mp4;base64,{mp4}" type="video/mp4">
</video>
'''))
print('\nVideo is playing above!')

In [ ]:
# Cell 7: Try different prompts (optional)
import torch

PROMPTS = [
    ('soaring through aurora borealis, cosmic sky, stars, '
     'golden wings glowing, majestic flight, fantasy art'),
    ('diving through clouds at high speed, motion blur, '
     'action pose, wind effects, epic angle, game cinematic'),
]

for i, prompt in enumerate(PROMPTS):
    generator = torch.manual_seed(42 + i)
    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        ip_adapter_image=ref_image,
        num_frames=16,
        guidance_scale=7.5,
        num_inference_steps=25,
        generator=generator,
        width=512,
        height=512,
    )
    out_path = f'/content/flying_v{i+2}.mp4'
    export_to_video(result.frames[0], out_path, fps=8)
    print(f'Variant {i+2}: {prompt[:50]}... -> {out_path}')

print('\nAll variants generated!')

In [ ]:
# Cell 8: Download all videos
from google.colab import files
import glob, os

for f in sorted(glob.glob('/content/flying_*.mp4')) + sorted(glob.glob('/content/flying_*.gif')):
    size = os.path.getsize(f) / 1024**2
    print(f'Downloading {os.path.basename(f)} ({size:.1f} MB)...')
    files.download(f)

print('Done!')